# WEEK 6 - Functions 2, 3, 4, 5, 6

## Exploitative Bayesian Optimization (GP + UCB with β=0) + Progress Validation

This notebook:
1. Uses exploitative Bayesian Optimization (β=0) for functions 2-6
2. **Validates progress**: Checks if we're improving and flags if stuck
3. Loads data from weeks 1-5
4. Saves results to `results/` directory

### STEP 0 - IMPORT RELEVANT PACKAGES

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

import sys
sys.path.append('../..')  # Go up two levels to project root

import importlib
import data_utils
importlib.reload(data_utils)

# Import utility functions
from data_utils import (
    load_initial_data,
    load_week1_data,
    load_week2_data,
    load_week3_data,
    load_week4_data,
    _load_latest_week_data,
    plot_2d_scatter_with_distribution,
    plot_3d_scatter_with_distribution,
    plot_tsne_with_distribution,
)

# Import the exploitative Bayesian Optimization module
from bayesian_optimization_exploitative import SimpleBayesianOptimization

TSNE_RANDOM_STATE = 42

### STEP 1 - LOAD INITIAL DATA

In [ ]:
# Load initial data using utility function
base_dir = Path('../../data/initial_data')
initial_data = load_initial_data(base_dir)

print(f"Loaded initial data for {len(initial_data)} functions")
for function_name, data in initial_data.items():
    print(f"  {function_name}: {data['inputs'].shape[0]} samples, {data['inputs'].shape[1]} dimensions")

### STEP 2 - LOAD WEEK 1 DATA

In [ ]:
# Load week1 data using utility function
week1_dir = Path('../../data/week1')
week1_data = load_week1_data(week1_dir)

print(f"Loaded week1 data for {len(week1_data)} functions")
for function_name, data in week1_data.items():
    print(f"  {function_name}: Input shape = {data['input'].shape}, Output = {data['output']}")

### STEP 3 - LOAD WEEK 2 DATA

In [ ]:
# Load week2 data using utility function
week2_dir = Path('../../data/week2')
week2_data = load_week2_data(week2_dir)

print(f"Loaded week2 data for {len(week2_data)} functions")
for function_name, data in week2_data.items():
    print(f"  {function_name}: Input shape = {data['input'].shape}, Output = {data['output']}")

### STEP 4 - LOAD WEEK 3 DATA

In [ ]:
# Load week3 data using utility function
week3_dir = Path('../../data/week3')
week3_data = load_week3_data(week3_dir)

print(f"Loaded week3 data for {len(week3_data)} functions")
for function_name, data in week3_data.items():
    print(f"  {function_name}: Input shape = {data['input'].shape}, Output = {data['output']}")

### STEP 5 - LOAD WEEK 4 DATA

In [ ]:
# Load week4 data using utility function
week4_dir = Path('../../data/week4')
week4_data = load_week4_data(week4_dir)

print(f"Loaded week4 data for {len(week4_data)} functions")
for function_name, data in week4_data.items():
    print(f"  {function_name}: Input shape = {data['input'].shape}, Output = {data['output']}")

### STEP 6 - LOAD WEEK 5 DATA

In [ ]:
# Load week5 data using utility function
week5_dir = Path('../../data/week5')
week5_data = _load_latest_week_data(week5_dir, 'Week 5')

print(f"Loaded week5 data for {len(week5_data)} functions")
for function_name, data in week5_data.items():
    print(f"  {function_name}: Input shape = {data['input'].shape}, Output = {data['output']}")

### VALIDATION FUNCTION - Check for Maximum Progress

In [ ]:
def validate_progress(y_observed, function_name, threshold_pct=5.0):
    """
    Check if we're making progress toward finding the maximum.
    
    Returns:
        status: 'improving', 'plateau', or 'stuck'
        message: Description of the status
        flag: True if we should be concerned
    """
    if len(y_observed) < 3:
        return 'insufficient_data', 'Not enough data to validate', False
    
    # Get last 3 observations
    recent = y_observed[-3:]
    best_overall = y_observed.max()
    best_recent = recent.max()
    
    # Check if recent best is close to overall best
    if best_recent >= best_overall * 0.99:  # Within 1% of best
        improvement_pct = ((best_recent - y_observed[-4]) / abs(y_observed[-4]) * 100) if len(y_observed) > 3 and y_observed[-4] != 0 else 0
        
        if improvement_pct < threshold_pct:
            return 'plateau', f'Best value found ({best_recent:.6f}) but recent improvement < {threshold_pct}%', True
        else:
            return 'improving', f'Making good progress, best = {best_recent:.6f}', False
    else:
        # Recent values not near best
        recent_trend = recent[-1] - recent[0]
        if recent_trend > 0:
            return 'improving', f'Recent trend positive, approaching best ({best_overall:.6f})', False
        else:
            return 'stuck', f'Recent values declining! Best={best_overall:.6f}, Recent={best_recent:.6f}', True
    
    return 'unknown', 'Unable to determine status', False

print('Validation function loaded.')

### FUNCTION 2 - EXPLOITATIVE BAYESIAN OPTIMIZATION

In [ ]:
# Function 2 - Combine all data (initial + weeks 1-5)
function_name = 'function_2'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get weekly new inputs and outputs
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']
week4_input = week4_data[function_name]['input']
week4_output = week4_data[function_name]['output']
week5_input = week5_data[function_name]['input']
week5_output = week5_data[function_name]['output']

# Combine all observed data
X_observed = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1),
    week4_input.reshape(1, -1),
    week5_input.reshape(1, -1)
])
y_observed = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output]),
    np.array([week4_output]),
    np.array([week5_output])
])

# Validate progress
status, message, flag = validate_progress(y_observed, function_name)

print("=" * 60)
print(f"FUNCTION 2 - COMBINED DATA & VALIDATION")
print("=" * 60)
print(f"\nCombined inputs shape: {X_observed.shape}")
print(f"Combined outputs shape: {y_observed.shape}")
print(f"\nBest observed output so far: {y_observed.max():.6f}")
print(f"Mean output: {y_observed.mean():.6f}")
print(f"Std output: {y_observed.std():.6f}")
print(f"\n{'⚠️ ' if flag else '✓ '}PROGRESS STATUS: {status.upper()}")
print(f"  {message}")
print("=" * 60)

# Visualize: show week 4 and week 5 points
plot_2d_scatter_with_distribution(
    original_inputs, original_outputs,
    week5_input, week5_output,
    function_name,
    prior_input=week4_input,
    prior_output=week4_output,
    prior_point_label="Week 4 point",
    new_point_label="Week 5 point",
    connect_points=True,
    connect_color="tab:orange"
)

In [ ]:
# Function 2 - Run Exploitative Bayesian Optimization
function_name = 'function_2'
n_dims = X_observed.shape[1]
bounds = [[0.0, 1.0]] * n_dims

# Create BO object with beta=0 for pure exploitation
bo = SimpleBayesianOptimization(bounds=bounds, n_initial=0, random_state=42)

# Fit GP to observed data
bo.fit(X_observed, y_observed)

# Suggest next point using UCB with beta=0.0 (pure exploitation)
next_point = bo.suggest_next_point(beta=0.0)
ucb_value = bo.acquisition_ucb(next_point.reshape(1, -1), beta=0.0)[0]

# Format for submission
formatted_point = '-'.join([f'{x:.6f}' for x in next_point])

print("\n" + "=" * 60)
print(f"FUNCTION 2 - EXPLOITATIVE BO SUGGESTION (β=0)")
print("=" * 60)
print(f"Suggested next point: {formatted_point}")
print(f"Expected value (UCB with β=0 = mean): {ucb_value:.6f}")
if flag:
    print(f"\n⚠️  WARNING: {message}")
    print("   Consider exploring or checking surrogate model quality")
print("=" * 60)

# Save to JSON
result_json = {
    "function_name": function_name,
    "next_point": next_point.tolist(),
    "formatted": formatted_point,
    "method": "exploitative_bayesian_optimization",
    "beta": 0.0,
    "expected_value": float(ucb_value),
    "validation_status": status,
    "validation_message": message,
    "validation_flag": flag
}

output_path = 'results/function2_next_point.json'
with open(output_path, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nResult saved to: {output_path}")

### FUNCTION 3 - EXPLOITATIVE BAYESIAN OPTIMIZATION

In [ ]:
# Function 3 - Combine all data (initial + weeks 1-5)
function_name = 'function_3'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get weekly new inputs and outputs
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']
week4_input = week4_data[function_name]['input']
week4_output = week4_data[function_name]['output']
week5_input = week5_data[function_name]['input']
week5_output = week5_data[function_name]['output']

# Combine all observed data
X_observed = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1),
    week4_input.reshape(1, -1),
    week5_input.reshape(1, -1)
])
y_observed = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output]),
    np.array([week4_output]),
    np.array([week5_output])
])

# Validate progress
status, message, flag = validate_progress(y_observed, function_name)

print("=" * 60)
print(f"FUNCTION 3 - COMBINED DATA & VALIDATION")
print("=" * 60)
print(f"\nCombined inputs shape: {X_observed.shape}")
print(f"Combined outputs shape: {y_observed.shape}")
print(f"\nBest observed output so far: {y_observed.max():.6f}")
print(f"Mean output: {y_observed.mean():.6f}")
print(f"Std output: {y_observed.std():.6f}")
print(f"\n{'⚠️ ' if flag else '✓ '}PROGRESS STATUS: {status.upper()}")
print(f"  {message}")
print("=" * 60)

# Visualize: show week 4 and week 5 points
plot_3d_scatter_with_distribution(
    original_inputs, original_outputs,
    week5_input, week5_output,
    function_name,
    prior_input=week4_input,
    prior_output=week4_output,
    prior_point_label="Week 4 point",
    new_point_label="Week 5 point",
    connect_points=True,
    connect_color="tab:orange"
)

In [ ]:
# Function 3 - Run Exploitative Bayesian Optimization
function_name = 'function_3'
n_dims = X_observed.shape[1]
bounds = [[0.0, 1.0]] * n_dims

# Create BO object with beta=0 for pure exploitation
bo = SimpleBayesianOptimization(bounds=bounds, n_initial=0, random_state=42)

# Fit GP to observed data
bo.fit(X_observed, y_observed)

# Suggest next point using UCB with beta=0.0 (pure exploitation)
next_point = bo.suggest_next_point(beta=0.0)
ucb_value = bo.acquisition_ucb(next_point.reshape(1, -1), beta=0.0)[0]

# Format for submission
formatted_point = '-'.join([f'{x:.6f}' for x in next_point])

print("\n" + "=" * 60)
print(f"FUNCTION 3 - EXPLOITATIVE BO SUGGESTION (β=0)")
print("=" * 60)
print(f"Suggested next point: {formatted_point}")
print(f"Expected value (UCB with β=0 = mean): {ucb_value:.6f}")
if flag:
    print(f"\n⚠️  WARNING: {message}")
    print("   Consider exploring or checking surrogate model quality")
print("=" * 60)

# Save to JSON
result_json = {
    "function_name": function_name,
    "next_point": next_point.tolist(),
    "formatted": formatted_point,
    "method": "exploitative_bayesian_optimization",
    "beta": 0.0,
    "expected_value": float(ucb_value),
    "validation_status": status,
    "validation_message": message,
    "validation_flag": flag
}

output_path = 'results/function3_next_point.json'
with open(output_path, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nResult saved to: {output_path}")

### FUNCTION 4 - EXPLOITATIVE BAYESIAN OPTIMIZATION

In [ ]:
# Function 4 - Combine all data (initial + weeks 1-5)
function_name = 'function_4'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get weekly new inputs and outputs
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']
week4_input = week4_data[function_name]['input']
week4_output = week4_data[function_name]['output']
week5_input = week5_data[function_name]['input']
week5_output = week5_data[function_name]['output']

# Combine all observed data
X_observed = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1),
    week4_input.reshape(1, -1),
    week5_input.reshape(1, -1)
])
y_observed = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output]),
    np.array([week4_output]),
    np.array([week5_output])
])

# Validate progress
status, message, flag = validate_progress(y_observed, function_name)

print("=" * 60)
print(f"FUNCTION 4 - COMBINED DATA & VALIDATION")
print("=" * 60)
print(f"\nCombined inputs shape: {X_observed.shape}")
print(f"Combined outputs shape: {y_observed.shape}")
print(f"\nBest observed output so far: {y_observed.max():.6f}")
print(f"Mean output: {y_observed.mean():.6f}")
print(f"Std output: {y_observed.std():.6f}")
print(f"\n{'⚠️ ' if flag else '✓ '}PROGRESS STATUS: {status.upper()}")
print(f"  {message}")
print("=" * 60)

# Visualize: show week 4 and week 5 points
plot_tsne_with_distribution(
    original_inputs, original_outputs,
    week5_input, week5_output,
    function_name,
    prior_input=week4_input,
    prior_output=week4_output,
    prior_point_label="Week 4 point",
    new_point_label="Week 5 point",
    connect_points=True,
    connect_color="tab:orange",\n    random_state=TSNE_RANDOM_STATE
)

In [ ]:
# Function 4 - Run Exploitative Bayesian Optimization
function_name = 'function_4'
n_dims = X_observed.shape[1]
bounds = [[0.0, 1.0]] * n_dims

# Create BO object with beta=0 for pure exploitation
bo = SimpleBayesianOptimization(bounds=bounds, n_initial=0, random_state=42)

# Fit GP to observed data
bo.fit(X_observed, y_observed)

# Suggest next point using UCB with beta=0.0 (pure exploitation)
next_point = bo.suggest_next_point(beta=0.0)
ucb_value = bo.acquisition_ucb(next_point.reshape(1, -1), beta=0.0)[0]

# Format for submission
formatted_point = '-'.join([f'{x:.6f}' for x in next_point])

print("\n" + "=" * 60)
print(f"FUNCTION 4 - EXPLOITATIVE BO SUGGESTION (β=0)")
print("=" * 60)
print(f"Suggested next point: {formatted_point}")
print(f"Expected value (UCB with β=0 = mean): {ucb_value:.6f}")
if flag:
    print(f"\n⚠️  WARNING: {message}")
    print("   Consider exploring or checking surrogate model quality")
print("=" * 60)

# Save to JSON
result_json = {
    "function_name": function_name,
    "next_point": next_point.tolist(),
    "formatted": formatted_point,
    "method": "exploitative_bayesian_optimization",
    "beta": 0.0,
    "expected_value": float(ucb_value),
    "validation_status": status,
    "validation_message": message,
    "validation_flag": flag
}

output_path = 'results/function4_next_point.json'
with open(output_path, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nResult saved to: {output_path}")

### FUNCTION 5 - EXPLOITATIVE BAYESIAN OPTIMIZATION

In [ ]:
# Function 5 - Combine all data (initial + weeks 1-5)
function_name = 'function_5'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get weekly new inputs and outputs
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']
week4_input = week4_data[function_name]['input']
week4_output = week4_data[function_name]['output']
week5_input = week5_data[function_name]['input']
week5_output = week5_data[function_name]['output']

# Combine all observed data
X_observed = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1),
    week4_input.reshape(1, -1),
    week5_input.reshape(1, -1)
])
y_observed = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output]),
    np.array([week4_output]),
    np.array([week5_output])
])

# Validate progress
status, message, flag = validate_progress(y_observed, function_name)

print("=" * 60)
print(f"FUNCTION 5 - COMBINED DATA & VALIDATION")
print("=" * 60)
print(f"\nCombined inputs shape: {X_observed.shape}")
print(f"Combined outputs shape: {y_observed.shape}")
print(f"\nBest observed output so far: {y_observed.max():.6f}")
print(f"Mean output: {y_observed.mean():.6f}")
print(f"Std output: {y_observed.std():.6f}")
print(f"\n{'⚠️ ' if flag else '✓ '}PROGRESS STATUS: {status.upper()}")
print(f"  {message}")
print("=" * 60)

# Visualize: show week 4 and week 5 points
plot_tsne_with_distribution(
    original_inputs, original_outputs,
    week5_input, week5_output,
    function_name,
    prior_input=week4_input,
    prior_output=week4_output,
    prior_point_label="Week 4 point",
    new_point_label="Week 5 point",
    connect_points=True,
    connect_color="tab:orange",\n    random_state=TSNE_RANDOM_STATE
)

In [ ]:
# Function 5 - Run Exploitative Bayesian Optimization
function_name = 'function_5'
n_dims = X_observed.shape[1]
bounds = [[0.0, 1.0]] * n_dims

# Create BO object with beta=0 for pure exploitation
bo = SimpleBayesianOptimization(bounds=bounds, n_initial=0, random_state=42)

# Fit GP to observed data
bo.fit(X_observed, y_observed)

# Suggest next point using UCB with beta=0.0 (pure exploitation)
next_point = bo.suggest_next_point(beta=0.0)
ucb_value = bo.acquisition_ucb(next_point.reshape(1, -1), beta=0.0)[0]

# Format for submission
formatted_point = '-'.join([f'{x:.6f}' for x in next_point])

print("\n" + "=" * 60)
print(f"FUNCTION 5 - EXPLOITATIVE BO SUGGESTION (β=0)")
print("=" * 60)
print(f"Suggested next point: {formatted_point}")
print(f"Expected value (UCB with β=0 = mean): {ucb_value:.6f}")
if flag:
    print(f"\n⚠️  WARNING: {message}")
    print("   Consider exploring or checking surrogate model quality")
print("=" * 60)

# Save to JSON
result_json = {
    "function_name": function_name,
    "next_point": next_point.tolist(),
    "formatted": formatted_point,
    "method": "exploitative_bayesian_optimization",
    "beta": 0.0,
    "expected_value": float(ucb_value),
    "validation_status": status,
    "validation_message": message,
    "validation_flag": flag
}

output_path = 'results/function5_next_point.json'
with open(output_path, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nResult saved to: {output_path}")

### FUNCTION 6 - EXPLOITATIVE BAYESIAN OPTIMIZATION

In [ ]:
# Function 6 - Combine all data (initial + weeks 1-5)
function_name = 'function_6'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get weekly new inputs and outputs
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']
week4_input = week4_data[function_name]['input']
week4_output = week4_data[function_name]['output']
week5_input = week5_data[function_name]['input']
week5_output = week5_data[function_name]['output']

# Combine all observed data
X_observed = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1),
    week4_input.reshape(1, -1),
    week5_input.reshape(1, -1)
])
y_observed = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output]),
    np.array([week4_output]),
    np.array([week5_output])
])

# Validate progress
status, message, flag = validate_progress(y_observed, function_name)

print("=" * 60)
print(f"FUNCTION 6 - COMBINED DATA & VALIDATION")
print("=" * 60)
print(f"\nCombined inputs shape: {X_observed.shape}")
print(f"Combined outputs shape: {y_observed.shape}")
print(f"\nBest observed output so far: {y_observed.max():.6f}")
print(f"Mean output: {y_observed.mean():.6f}")
print(f"Std output: {y_observed.std():.6f}")
print(f"\n{'⚠️ ' if flag else '✓ '}PROGRESS STATUS: {status.upper()}")
print(f"  {message}")
print("=" * 60)

# Visualize: show week 4 and week 5 points
plot_tsne_with_distribution(
    original_inputs, original_outputs,
    week5_input, week5_output,
    function_name,
    prior_input=week4_input,
    prior_output=week4_output,
    prior_point_label="Week 4 point",
    new_point_label="Week 5 point",
    connect_points=True,
    connect_color="tab:orange",\n    random_state=TSNE_RANDOM_STATE
)

In [ ]:
# Function 6 - Run Exploitative Bayesian Optimization
function_name = 'function_6'
n_dims = X_observed.shape[1]
bounds = [[0.0, 1.0]] * n_dims

# Create BO object with beta=0 for pure exploitation
bo = SimpleBayesianOptimization(bounds=bounds, n_initial=0, random_state=42)

# Fit GP to observed data
bo.fit(X_observed, y_observed)

# Suggest next point using UCB with beta=0.0 (pure exploitation)
next_point = bo.suggest_next_point(beta=0.0)
ucb_value = bo.acquisition_ucb(next_point.reshape(1, -1), beta=0.0)[0]

# Format for submission
formatted_point = '-'.join([f'{x:.6f}' for x in next_point])

print("\n" + "=" * 60)
print(f"FUNCTION 6 - EXPLOITATIVE BO SUGGESTION (β=0)")
print("=" * 60)
print(f"Suggested next point: {formatted_point}")
print(f"Expected value (UCB with β=0 = mean): {ucb_value:.6f}")
if flag:
    print(f"\n⚠️  WARNING: {message}")
    print("   Consider exploring or checking surrogate model quality")
print("=" * 60)

# Save to JSON
result_json = {
    "function_name": function_name,
    "next_point": next_point.tolist(),
    "formatted": formatted_point,
    "method": "exploitative_bayesian_optimization",
    "beta": 0.0,
    "expected_value": float(ucb_value),
    "validation_status": status,
    "validation_message": message,
    "validation_flag": flag
}

output_path = 'results/function6_next_point.json'
with open(output_path, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nResult saved to: {output_path}")

### SUMMARY - ALL FUNCTIONS WITH VALIDATION

In [ ]:
# Print summary of all suggestions with validation flags
print("=" * 60)
print("EXPLOITATIVE BAYESIAN OPTIMIZATION - SUMMARY")
print("Functions 2, 3, 4, 5, 6")
print("=" * 60)
print("\nMethod: Gaussian Process with UCB (β=0, pure exploitation)")
print("\nNext points for submission:")
print()

validation_summary = []
for func_num in range(2, 7):
    json_path = f'results/function{func_num}_next_point.json'
    if Path(json_path).exists():
        with open(json_path, 'r') as f:
            result = json.load(f)
            flag_symbol = '⚠️ ' if result.get('validation_flag', False) else '✓ '
            print(f"{flag_symbol}function_{func_num}: {result['formatted']}")
            print(f"  Expected value: {result['expected_value']:.6f}")
            print(f"  Status: {result.get('validation_status', 'unknown').upper()}")
            if result.get('validation_flag', False):
                validation_summary.append(f"function_{func_num}")
    else:
        print(f"function_{func_num}: [NOT GENERATED]")

print()
print("=" * 60)
if validation_summary:
    print(f"\n⚠️  ATTENTION: {len(validation_summary)} function(s) flagged for review:")
    for fn in validation_summary:
        print(f"  - {fn}")
    print("\nConsider:")
    print("  - Increasing exploration (β > 0)")
    print("  - Checking GP surrogate quality")
    print("  - Reviewing recent data points")
else:
    print("\n✓ All functions showing healthy progress!")